In [ ]:
# =========================================
# 📦 1. Setup & Data Loading
# =========================================

import pandas as pd
import openpyxl
from IPython.display import display

# Path to cleaned dataset (use relative path in production)
file_path = r"../data/edited/online_retail_cleaned.xlsx"

# Load dataset
df = pd.read_excel(file_path, engine='openpyxl')


# =========================================
# 📊 2. RFM Analysis (Recency, Frequency, Monetary)
# =========================================

# Define snapshot date (last transaction + 1 day)
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Create RFM table
rfm = (
    df.groupby('CustomerID')
      .agg(
          Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
          Frequency=('InvoiceNo', 'nunique'),
          Monetary=('Revenue', 'sum')
      )
      .reset_index()
)

rfm.head()


# =========================================
# 🔢 3. RFM Scoring (Quartiles)
# =========================================

rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])

rfm['F_score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    4,
    labels=[1,2,3,4]
)

rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

# Combine scores
rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

rfm.head()


# =========================================
# 🧩 4. Customer Segmentation
# =========================================

def segment_customer(row):
    if row['RFM_Score'] == '444':
        return 'Champions'
    elif row['R_score'] >= 3 and row['F_score'] >= 3:
        return 'Loyal Customers'
    elif row['R_score'] >= 3:
        return 'Recent Customers'
    elif row['F_score'] >= 3:
        return 'Frequent Customers'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

display(rfm['Segment'].value_counts())
rfm['Segment'].value_counts(normalize=True)


# =========================================
# 💰 5. Revenue by Segment
# =========================================

segment_revenue = (
    rfm.groupby('Segment')['Monetary']
        .sum()
        .sort_values(ascending=False)
)

display(segment_revenue)
segment_revenue / segment_revenue.sum()


# =========================================
# 💎 6. Customer Lifetime Value (CLV)
# =========================================

# Customer-level aggregation
customer_clv = (
    df.groupby('CustomerID')
      .agg(
          FirstPurchase=('InvoiceDate', 'min'),
          LastPurchase=('InvoiceDate', 'max'),
          Orders=('InvoiceNo', 'nunique'),
          TotalRevenue=('Revenue', 'sum')
      )
      .reset_index()
)

# Lifespan calculation
customer_clv['Lifespan_days'] = (
    customer_clv['LastPurchase'] - customer_clv['FirstPurchase']
).dt.days

customer_clv['Lifespan_months'] = customer_clv['Lifespan_days'] / 30


# =========================================
# ⚙️ 7. CLV Feature Engineering
# =========================================

# Avoid zero lifespan
customer_clv['Lifespan_months'] = customer_clv['Lifespan_months'].replace(0, 1)
customer_clv['Lifespan_days'] = customer_clv['Lifespan_days'].replace(0, 1)

# AOV & Purchase Frequency
customer_clv['AOV'] = (
    customer_clv['TotalRevenue'] / customer_clv['Orders']
)

customer_clv['Purchase_Frequency'] = (
    customer_clv['Orders'] / customer_clv['Lifespan_months']
)


# =========================================
# 📈 8. CLV Calculation
# =========================================

# Historical CLV
customer_clv['CLV'] = (
    customer_clv['AOV'] *
    customer_clv['Purchase_Frequency'] *
    customer_clv['Lifespan_months']
)

# Normalized CLV (removes lifespan bias)
avg_lifespan = customer_clv['Lifespan_months'].mean()

customer_clv['CLV_normalized'] = (
    customer_clv['AOV'] *
    customer_clv['Purchase_Frequency'] *
    avg_lifespan
)


# =========================================
# 🔗 9. Merge CLV with Segments
# =========================================

customer_clv = customer_clv.merge(
    rfm[['CustomerID', 'Segment']],
    on='CustomerID'
)

# CLV by segment
customer_clv.groupby('Segment')['CLV_normalized'].mean().sort_values(ascending=False)


# =========================================
# 💾 10. Export Final Dataset
# =========================================

customer_clv.to_excel(
    r"../data/edited/final_customer_dataset.xlsx",
    index=False
)

# CLV distribution overview
display(customer_clv['CLV_normalized'].describe())

# Top customers by CLV
customer_clv.sort_values('CLV_normalized', ascending=False).head(10)